In [263]:
import numpy as np
import pandas as pd

class topo2rest():
   def __init__(self, ifile:str, temps:list = [300.0, 500.0], nreps:int = 20, kappa:float = 1.00):
      '''Convert processed topology to REST2/3 input topology
         E_tot = gamma*E^{pp} + sqrt(gamma)*E^{pw} + E^{ww}
         gamma = T_0/T_i
         for REST2/3 bonds and angles are not scaled per the REST2 paper,
         however, for REST2 LJ parameters epsilon_i is scaled by epsilon_i*gamma
         for tempered atoms and all others are unmodified. 
         In the case of REST3, with the additon of sqrt(gamma)*kappa*E^{pw} which differs 
         from sqrt(gamma)*E^{pw}, we produced the combination rule 2 nonbonded terms
         involving protein-water interactions to override the pw nonbonded interactions
         as including gamma*kappa with each hot atom epsilon_i would result in the 
         incorrect form of: E_tot = gamma*kappa*E^{pp} + sqrt(gamma*kappa)*E^{pw} + E^{ww}:
         rather than the correct form: 
         E_tot = gamma*kappa*E^{pp} + sqrt(gamma)*kappa*E^{pw} + E^{ww}
         input
         ifile = inputtopology.top
         temps = ["lower temp":float, "upper temp":float ]; temperature range of replicas
         kappa:float = kappa scaling; if not equal to 1 REST3 implementation active
         hot_m = hot molecule; [0] for most systems will select the protein'''

      self.nreps = nreps
      self.kappa = kappa
      self.hot_m = None
      self.tempreps = self.compute_temperatures(temps)
      self.lambdai = self.compute_lambda()
      self.sections = {}
      self.sections_out = {}
      self.hard_order_sections = [ "defaults", "atomtypes", "nonbond_params", "bondtypes", \
                                   "constrainttypes", "angletypes", "dihedraltypes" , "moleculetype"]
      with open(ifile) as topo:
         self.readfile = topo.readlines()
      self.gather_param_sections()
         
   #def get_molecule_atoms(self):
   def compute_lambda(self):
      return [ Ti/self.tempreps[0] for Ti in self.tempreps ]
   
   def compute_temperatures(self, temp_range:list):
      from numpy import log, exp
      tlow, thigh = temp_range
      temps = []
      for i in range(self.nreps):
         temps.append(tlow*exp((i)*log(thigh/tlow)/(self.nreps-1)))
      return temps
   
   def parse_section(self,trunks):
      first_round = True
      output = []
      for line in self.readfile[trunks:]:
         if "[" not in line and first_round!=True:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output

   def get_molecule_atomtypes(self):
      import pandas as pd
      b=[]
      for i in self.sections['atomtypes']:
         i.split()
         if len(i.split())>1 and i.split()[0]!=';' and i.split()[0]!='[' :
            b.append(i.split())
      dataset = pd.DataFrame(b, columns=self.sections['atomtypes'][1][2:].split())
      print(dataset)
      
      
   # , hot_m:list = [0]
   
   
#   def show_molecule_names(self):
#      try:
#         pass
         

   def get_molecule_atoms(self):
      import pandas as pd
      b=[]
      for i in self.sections['atoms']:
         i.split()
         if len(i.split())>1 and i.split()[0]!=';' and i.split()[0]!='[' :
            b.append(i.split())
   
   def _moleculetype_sub(self,linestart:int):
      first_round = True
      output = []
      for line in self.readfile[linestart:]:
         if "[" not in line and ';' not in line[:3]:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output 
   
   def identify_moltype_sections(self,trunks:int):
      section_start = []
      for i, line in enumerate(self.readfile[trunks:]):
         if '[' in line and 'moleculetype' not in line and 'system' not in line:
            section_start.append(i+trunks)
         elif i != 0 and 'moleculetype' in line or 'system' in line:
            break
      return section_start

   def parse_moleculetypes(self,trunks:int):
      first_round = True
      output = {}
      sections = self.identify_moltype_sections(trunks)
      print(sections)
      output['header'] = self.readfile[trunks:trunks+2]
      for section in sections:
         section_ = self.readfile[section].split()[1]
         output[section_] = self._moleculetype_sub(section)
      return output
   
   def get_scale_dehedrals_(self):
      import pandas as pd
      # need to get atom reference for which parameters to scale 
      dataset = pd.DataFrame([i.split()[:-2] if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
                               if '\n' not in i[:3]], columns=self.sections['dihedraltypes'][1][1:].split())
      print(dataset)
      self.sections[self.hard_order_sections[-1]]
      
      #TODO apply scaling with gamma and kappa 
      #     if kappa = 1.0 don't print extra nonbonded terms
      
      
   def get_scale_nonbonded(self, gamma:float, kappa:float):
      import pandas as pd
      atom_es = None
      # TODO everything

   def gather_param_sections(self):
      for section in self.hard_order_sections[:-1]:
         is_select = [ i for i, line in enumerate(self.readfile) if section in line ]
         in_select = [f' [ {section} ] \n']
         for i in is_select:
            in_select += self.parse_section(i)
         self.sections[section]=in_select
      section = self.hard_order_sections[-1]
      is_select = [ i for i, line in enumerate(self.readfile) if section in line ] 
      moltype_dict = {} 
      for i in range(len(is_select)):
         moltype_dict[i] = self.parse_moleculetypes(is_select[i])
      self.sections[section] = moltype_dict
         

In [264]:
test = topo2rest('./example_topo/processed.top')

[965, 1636, 2272, 3880, 4994, 6638]
[7039, 7065, 7089, 7143, 7184]
[7254, 7261, 7265, 7269]
[7281]
[7290]
[7299]
[7308]
[7317]
[7326]
[7335]
[7344]
[7353]
[7362]


In [278]:
test.readfile[7269]

'[ exclusions ]\n'

In [279]:
test.sections['moleculetype']

{0: {'header': ['[ moleculetype ]\n', '; Name            nrexcl\n'],
  'atoms': ['     1         N3      1    GLY      N      1     0.2943      14.01\n',
   '     2          H      1    GLY     H1      2     0.1642      1.008\n',
   '     3          H      1    GLY     H2      3     0.1642      1.008\n',
   '     4          H      1    GLY     H3      4     0.1642      1.008\n',
   '     5         C1      1    GLY     CA      5      -0.01      12.01\n',
   '     6         HP      1    GLY    HA1      6     0.0895      1.008\n',
   '     7         HP      1    GLY    HA2      7     0.0895      1.008\n',
   '     8          C      1    GLY      C      8     0.6163      12.01\n',
   '     9         OB      1    GLY      O      9    -0.5722         16   ; qtot 1\n',
   '    10          N      2    MET      N     10    -0.4157      14.01\n',
   '    11         HB      2    MET      H     11     0.2719      1.008\n',
   '    12         CT      2    MET     CA     12    -0.0237      12.01\n',

In [226]:
test.gather_param_sections()

In [214]:
test.get_scale_dehedrals_()

      i   j   k   l func  phase        kd pn
0     C  C1   N   O    4  180.0   4.60240  2
1     C  C1   N  OB    4  180.0   4.60240  2
2     C  C1   N   H    4  180.0   4.60240  2
3     C  C1   N  HB    4  180.0   4.60240  2
4    C9   O   C  OH    4  180.0  43.93200  2
..   ..  ..  ..  ..  ...    ...       ... ..
327   X  C*  CT   X    9    0.0   0.00000  0
328   X  C7   N   X    9    0.0   0.00000  0
329   X   C  C7   X    9    0.0   0.00000  0
330   X  C5  CT   X    9    0.0   0.00000  0
331   X  C6  CT   X    9    0.0   0.00000  0

[332 rows x 8 columns]


In [220]:
test.sections['dihedraltypes']

AttributeError: 'list' object has no attribute 'keys'

In [155]:
test.sections['moleculetype']

[' [ moleculetype ] \n',
 '; Name            nrexcl\n',
 'Protein_chain_A     3\n',
 '\n',
 '; Name            nrexcl\n',
 'HxD          3\n',
 '\n',
 '; molname\tnrexcl\n',
 'SOL\t\t1\n',
 '\n',
 '; molname       nrexcl\n',
 'IB+             1       ; big positive ion\n',
 '\n',
 '; molname       nrexcl\n',
 'CA              1\n',
 '\n',
 '; molname       nrexcl\n',
 'CL              1\n',
 '\n',
 '; molname       nrexcl\n',
 'NA              1\n',
 '\n',
 '; molname       nrexcl\n',
 'MG              1\n',
 '\n',
 '; molname       nrexcl\n',
 'K               1\n',
 '\n',
 '; molname       nrexcl\n',
 'RB              1\n',
 '\n',
 '; molname       nrexcl\n',
 'CS              1\n',
 '\n',
 '; molname       nrexcl\n',
 'LI              1\n',
 '\n',
 '; molname       nrexcl\n',
 'ZN              1\n',
 '\n']

In [157]:
f=open('test_dihedraltypes.txt','w')
f.writelines(test.sections['dihedraltypes'])
f.close()
